# 05 — Evaluation & Comparison
Load all trained models and compare them across metrics.

In [ ]:
# --- Google Colab Setup ---
import os

if os.getenv('COLAB_RELEASE_TAG'):
    !git clone https://github.com/lmartim4/CSC-52081-ContinousMountainCar.git /content/repo
    %cd /content/repo
    !pip install -e ".[notebook]"
else:
    os.chdir(os.path.join(os.path.dirname(os.path.abspath("__file__")), '..'))

In [ ]:
import pandas as pd
from stable_baselines3 import DQN, PPO
from src.wrappers import make_pixel_env, make_symbolic_env
from src.utils.evaluation import evaluate_agent
from src.config import EVAL_DEFAULTS

In [ ]:
# Load models
models = {
    'Pixel DQN': (DQN.load('models/pixel_dqn/final_model'), make_pixel_env),
    'Pixel PPO': (PPO.load('models/pixel_ppo/final_model'), make_pixel_env),
    'Symbolic DQN': (DQN.load('models/symbolic_dqn/final_model'), make_symbolic_env),
    'Symbolic PPO': (PPO.load('models/symbolic_ppo/final_model'), make_symbolic_env),
}

In [ ]:
# Evaluate all models
all_results = {}
for name, (model, env_fn) in models.items():
    print(f'Evaluating {name}...')
    env = env_fn()
    results = evaluate_agent(model, env, n_episodes=EVAL_DEFAULTS.n_eval_episodes)
    all_results[name] = results
    print(f"  Mean reward: {results['mean_reward']:.1f} ± {results['std_reward']:.1f}")
    print(f"  Flag rate: {results['flag_rate']:.2%}")
    env.close()

In [ ]:
# Summary table
df = pd.DataFrame({
    name: {
        'Mean Reward': f"{r['mean_reward']:.1f} ± {r['std_reward']:.1f}",
        'Mean Length': f"{r['mean_length']:.0f}",
        'Flag Rate': f"{r['flag_rate']:.1%}",
    }
    for name, r in all_results.items()
}).T
df

In [ ]:
# Save results
df.to_csv('results/evaluation_summary.csv')
print('Results saved to results/evaluation_summary.csv')